# Phase 06 — Topology Builder

> **📁 Résultat de cette phase :** La topologie réseau est sauvegardée dans `research/config/network_topology_1024.yaml` et les mappages cellules-antennes dans `data/processed/cell_antenna_map_1024.json`.

Ce notebook construit la structure spatiale du réseau sur la grille 100×100 de Milan (~181 km²).

## 📡 Calibration sur données réelles (ARPA Lombardia 2025)
Le déploiement est calibré sur la réalité du terrain milanais :
- **Centre (Duomo/Garibaldi)** : 3–4 antennes/km² (haute densité micro/smallcells).
- **Ceinture urbaine** : 1,5 antennes/km² (mix macro/micro).
- **Périphérie** : 0,3 antennes/km² (majorité macro).

## 📐 Capacité Radio Physique (Shannon)
Les capacités sont calculées via la formule de Shannon :
`C = BW * log2(1 + SINR) * eff_spectrale`
Nous distinguons deux types de capacités :
- **Capacité radio physique (capacity_mo)** : Seuil dur lié au matériel. Si le trafic dépasse ce plafond, il y a perte de paquets.
- **Capacité nominale (nominal_capacity)** : Seuil statistique (percentile 90 historique) qui déclenche l'optimisation SON. C'est le seuil de confort.

## 💡 Ontologie : Qui est qui ?
- **Cellule géographique** : Zone discrète de 235m × 235m (grille).
- **Antenne physique** : Équipement radio réel (macro/micro/smallcell) avec une capacité Shannon propre.
- **Assignation** : Chaque cellule est assignée à l'antenne dont le centre est le plus proche (Voronoi discret).

In [ ]:
import numpy as np
import json
import yaml
import polars as pl
from pathlib import Path
from scipy.spatial import cKDTree
from collections import defaultdict

GRID_SIZE = 100
SEED = 42
rng = np.random.default_rng(SEED)

ANTENNA_PROFILES = {
    'macro':     {'bw_mhz': 20, 'sinr_db': 15, 'tx_power_dBm': 46, 'height_m': 25, 'coverage_radius_grid': 3.5},
    'micro':     {'bw_mhz': 20, 'sinr_db': 20, 'tx_power_dBm': 37, 'height_m': 8,  'coverage_radius_grid': 0.9},
    'smallcell': {'bw_mhz': 10, 'sinr_db': 25, 'tx_power_dBm': 24, 'height_m': 4,  'coverage_radius_grid': 0.2},
}

DENSITY_ZONES = [
    {'name': 'center',      'rows': (40, 60), 'cols': (40, 60), 'density': 0.22,  'type_proba': {'macro': 0.3, 'micro': 0.45, 'smallcell': 0.25}, 'jitter': 2.0},
    {'name': 'urban_ring', 'rows': (20, 80), 'cols': (20, 80), 'density': 0.06,  'type_proba': {'macro': 0.55, 'micro': 0.30, 'smallcell': 0.15}, 'jitter': 3.5},
    {'name': 'periphery',  'rows': (0, 100), 'cols': (0, 100), 'density': 0.012, 'type_proba': {'macro': 0.85, 'micro': 0.12, 'smallcell': 0.03}, 'jitter': 5.0},
]

def capacity_from_radio(bw_mhz, sinr_db, spectral_eff=0.6, utilization=0.6, duration_s=1800):
    bw_hz     = bw_mhz * 1e6
    sinr_lin  = 10 ** (sinr_db / 10)
    cap_bps   = bw_hz * np.log2(1 + sinr_lin) * spectral_eff
    volume_mo = (cap_bps * duration_s * utilization) / (8 * 1e6)
    return round(volume_mo, 1)

def generate_milan_topology(grid_size=100, rng=None):
    if rng is None: rng = np.random.default_rng(42)
    antennas = {}
    idx = 0
    placed_positions = []
    for zone in DENSITY_ZONES:
        r0, r1 = zone['rows']; c0, c1 = zone['cols']
        area = (r1 - r0) * (c1 - c0)
        n_ant = int(area * zone['density'])
        for _ in range(n_ant):
            if zone['name'] == 'center':
                r = rng.normal((r0+r1)/2, (r1-r0)/6)
                c = rng.normal((c0+c1)/2, (c1-c0)/6)
            else:
                r = rng.uniform(r0, r1); c = rng.uniform(c0, c1)
            r = float(np.clip(r + rng.normal(0, zone['jitter']*0.1), 0, grid_size-1))
            c = float(np.clip(c + rng.normal(0, zone['jitter']*0.1), 0, grid_size-1))
            
            too_close = any(np.hypot(r-pr, c-pc) < 0.5 for pr, pc in placed_positions)
            if too_close: continue
            
            types = list(zone['type_proba'].keys())
            probas = list(zone['type_proba'].values())
            ant_type = rng.choice(types, p=probas)
            profile = ANTENNA_PROFILES[ant_type]
            
            base_capacity = capacity_from_radio(profile['bw_mhz'], profile['sinr_db'])
            capacity_mo = float(base_capacity * (1 + rng.normal(0, 0.1)))
            
            antennas[f'A{idx:03d}'] = {
                'x': float(c), 'y': float(r), 'type': str(ant_type), 'zone': str(zone['name']),
                'capacity_mo': float(round(capacity_mo, 1)),
                'bw_mhz': int(profile['bw_mhz']), 'sinr_db': int(profile['sinr_db']),
                'tx_power_dBm': int(profile['tx_power_dBm']), 'height_m': int(profile['height_m']),
                'coverage_radius': float(profile['coverage_radius_grid']),
            }
            placed_positions.append((r, c)); idx += 1
    return antennas

antennas = generate_milan_topology(GRID_SIZE, rng)
print(f"{len(antennas)} antennes générées.")

## Assignation et Graphe de Voisinage
Chaque cellule géographique (square_id) est assignée à l'antenne la plus proche (Voronoi discret).

In [ ]:
def build_neighbor_graph(antennas, threshold=8.0):
    ant_ids = list(antennas.keys())
    coords = np.array([[antennas[a]['y'], antennas[a]['x']] for a in ant_ids])
    tree = cKDTree(coords)
    neighbors = {}
    for i, a_id in enumerate(ant_ids):
        idx_neighbors = tree.query_ball_point(coords[i], r=threshold)
        neighbors[a_id] = [ant_ids[j] for j in idx_neighbors if ant_ids[j] != a_id]
    return neighbors

def assign_all_cells(grid_size, antennas):
    all_cells = [(i, j) for i in range(grid_size) for j in range(grid_size)]
    cell_centers = np.array([(i + 0.5, j + 0.5) for i, j in all_cells])
    ant_ids = list(antennas.keys())
    ant_coords = np.array([[antennas[a]['y'], antennas[a]['x']] for a in ant_ids])
    tree = cKDTree(ant_coords)
    _, nearest_idx = tree.query(cell_centers, k=1)
    coverage = defaultdict(list)
    for (i, j), ant_idx in zip(all_cells, nearest_idx):
        square_id = i * grid_size + j + 1
        coverage[ant_ids[ant_idx]].append(int(square_id))
    return dict(coverage)

neighbor_graph = build_neighbor_graph(antennas)
full_coverage = assign_all_cells(GRID_SIZE, antennas)

# Extraction 600 cellules
work_600 = pl.read_parquet('../data/processed/work_600cells.parquet')['square_id'].unique().to_list()
work_600_set = set(work_600)
coverage_600 = {a: [c for c in cells if c in work_600_set] for a, cells in full_coverage.items()}
coverage_600 = {a: c for a, c in coverage_600.items() if c}

print(f"Assignation terminée. {len(coverage_600)} antennes couvrent les 600 cellules cibles.")

## Sauvegarde des configurations
On génère les fichiers nécessaires pour le simulateur spatial (Phase 8) et le MILP (Phase 9).

In [ ]:
Path('../config').mkdir(exist_ok=True)
with open('../config/network_topology.yaml', 'w') as f:
    yaml.dump({'antennas': antennas}, f)
with open('../data/processed/cell_antenna_map_600.json', 'w') as f:
    json.dump(coverage_600, f)
with open('../data/processed/neighbor_graph.json', 'w') as f:
    json.dump(neighbor_graph, f)
print("Fichiers sauvegardés.")